# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kratosontren/flyrank-ml-work/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# Two Paper Findings + My Methodology Questions

## Finding 1 – The Content Performance Curve

The paper reports that content performs best around 61–90 days after publication and tends to decline after approximately 270 days. It also notes that older pages can recover after being refreshed, but cautions that this should not be interpreted as natural recovery because refreshed pages likely contribute to the improvement. :contentReference[oaicite:0]{index=0}

**Where does the label come from?**

The comparison is based on observed historical search metrics such as health score, impressions and average position rather than manually assigned labels.

**Does the validation design support the claim?**

The evidence is observational rather than experimental. The comparisons are useful for decision-support, but they do not prove that refreshing content alone causes better performance.

---

## Finding 2 – Click Capture by Position Tier

The paper shows that weighted CTR decreases as average search position becomes worse, with the highest click capture occurring in the Top-3 positions. The paper also explains why weighted CTR is used instead of simple averages. :contentReference[oaicite:1]{index=1}

**Where does the label come from?**

The result comes directly from observed Search Console metrics (clicks and impressions) aggregated by ranking position.

**Does the validation design support the claim?**

The portfolio comparisons support a directional relationship between ranking position and CTR. However, they do not establish causation because the study is not a controlled experiment.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Honest Validation

The Week-5 Logistic Regression model was first evaluated using a standard random train-test split.

To better estimate how the model would generalize, I also evaluated it using a grouped split based on client identifiers so that pages from the same client did not appear in both training and testing sets.

The grouped split produced lower Accuracy, Precision, and F1 than the random split, while Recall remained high. This suggests that the random split gives a more optimistic estimate, whereas the grouped split provides a stricter and more realistic evaluation of model generalization.

These results should be interpreted as observed performance on this historical dataset rather than guaranteed production performance.

In [8]:
from datasets import load_dataset
import pandas as pd
from itertools import islice

# Load a working sample from the warehouse
daily = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)

sample = pd.DataFrame(list(islice(daily, 5000)))

print(sample.shape)
sample.head()

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

(5000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


In [9]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# ----------------------------------------------------
# Build the proxy target
# ----------------------------------------------------

threshold = sample["gsc_avg_position"].median()

sample["target"] = (
    sample["gsc_avg_position"] > threshold
).astype(int)

# ----------------------------------------------------
# IMPORTANT:
# Do NOT include gsc_avg_position because
# it is used to create the target.
# ----------------------------------------------------

feature_columns = [
    "gsc_impressions",
    "gsc_clicks"
]

X = sample[feature_columns].fillna(0)

y = sample["target"]

groups = sample["client_hash_id"]

# ----------------------------------------------------
# Random Split
# ----------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = LogisticRegression(max_iter=1000)

random_model.fit(X_train, y_train)

random_pred = random_model.predict(X_test)

# ----------------------------------------------------
# Grouped Split
# ----------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)

X_train_g = X.iloc[train_idx]
X_test_g = X.iloc[test_idx]

y_train_g = y.iloc[train_idx]
y_test_g = y.iloc[test_idx]

group_model = LogisticRegression(max_iter=1000)

group_model.fit(
    X_train_g,
    y_train_g
)

group_pred = group_model.predict(
    X_test_g
)

# ----------------------------------------------------
# Compare
# ----------------------------------------------------

results = pd.DataFrame({

    "Metric":[
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ],

    "Random Split":[

        accuracy_score(
            y_test,
            random_pred
        ),

        precision_score(
            y_test,
            random_pred,
            zero_division=0
        ),

        recall_score(
            y_test,
            random_pred,
            zero_division=0
        ),

        f1_score(
            y_test,
            random_pred,
            zero_division=0
        )

    ],

    "Grouped Split":[

        accuracy_score(
            y_test_g,
            group_pred
        ),

        precision_score(
            y_test_g,
            group_pred,
            zero_division=0
        ),

        recall_score(
            y_test_g,
            group_pred,
            zero_division=0
        ),

        f1_score(
            y_test_g,
            group_pred,
            zero_division=0
        )

    ]

})

results

,Metric,Random Split,Grouped Split
0,Accuracy,0.552000,0.349112
1,Precision,0.528261,0.335366
2,Recall,0.972000,0.982143
3,F1,0.684507,0.500000


# Leakage Audit

The final feature set was reviewed for potential leakage.

Checks performed:

- No future outcome variables were used.
- No label-derived columns were included.
- The feature used to create the proxy target (gsc_avg_position) was excluded from the model inputs.
- Only historical search signals available before the prediction point were used.

The reduction in performance under grouped validation also provides evidence that the model is not relying on obvious information leakage.

In [10]:
leakage_checks = pd.DataFrame({

    "Check":[
        "Future outcome columns",
        "Label-derived features",
        "Prediction-time available",
        "Target feature excluded"
    ],

    "Result":[
        "No",
        "No",
        "Yes",
        "Yes"
    ]

})

leakage_checks

,Check,Result
0,Future outcome columns,No
1,Label-derived features,No
2,Prediction-time available,Yes
3,Target feature excluded,Yes


# Claim Rewrite

### Original claim

The model predicts which pages should be refreshed.

### Rewritten claim

In this historical sample, the model identified pages whose historical search signals were directionally associated with the chosen proxy target.

Performance was lower under grouped validation than under a random split, indicating that results are sensitive to the evaluation design. Therefore, these findings should be interpreted as decision-support rather than proof that the model will accurately identify future refresh opportunities.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit my repo URL on the card. Done.